In [9]:
import polars as pl

from data_engineering.openairframes.read import scan_latest_registration_icao_for_join

EUROCONTROL_PATH = "/Volumes/T2-SSD/planequery/data/raw/eurocontrol/202406/intermediate/flights_20240601_20240630_time_columns.parquet"

df_eurocontrol = pl.read_parquet(EUROCONTROL_PATH)

In [10]:
COLUMNS = ["AC Registration", "ADEP", "ADES", "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME"]
REGISTRATION_JOIN_KEY = "registration_join_key"

df_openairframes = (
    scan_latest_registration_icao_for_join(real_icao_column="icao")
    .with_columns(
        pl.col("registration")
        .str.replace_all(r"[^A-Z0-9]", "")
        .alias(REGISTRATION_JOIN_KEY)
    )
    .filter(pl.col(REGISTRATION_JOIN_KEY) != "")
    .drop("registration")
    .unique(subset=[REGISTRATION_JOIN_KEY], keep="last", maintain_order=False)
)

df = (
    df_eurocontrol.select(COLUMNS)
    .with_columns(
        pl.col("AC Registration")
        .str.strip_chars()
        .str.to_uppercase()
        .str.replace_all(r"[^A-Z0-9]", "")
        .alias(REGISTRATION_JOIN_KEY)
    )
    .lazy()
    .join(df_openairframes, on=REGISTRATION_JOIN_KEY, how="left")
    .drop(REGISTRATION_JOIN_KEY)
    .select(["AC Registration", "icao", "ADEP", "ADES", "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME"])
    .collect()
)
df

AC Registration,icao,ADEP,ADES,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME
str,str,str,str,datetime[μs],datetime[μs]
"""N588TN""","""a79556""","""KPBI""","""EGLF""",2024-06-01 00:43:00,2024-06-01 08:53:00
"""N44WG""","""a54a3f""","""KFLL""","""LIMJ""",2024-06-01 00:09:00,2024-06-01 09:28:00
"""N686UA""","""a91b44""","""KEWR""","""EGLL""",2024-06-01 00:55:00,2024-06-01 07:50:46
"""N24973""","""a254d7""","""KLAX""","""EGLL""",2024-06-01 00:45:00,2024-06-01 10:38:57
"""9HVJO""","""4d20f7""","""KOPF""","""EGLF""",2024-06-01 00:47:00,2024-06-01 08:59:00
…,…,…,…,…,…
"""PKLEW""","""8a09df""","""GMMN""","""OEJN""",2024-07-01 00:21:42,2024-07-01 06:05:38
"""N761AJ""","""aa4548""","""KDFW""","""EIDW""",2024-07-01 01:18:00,2024-07-01 09:04:15
"""YIARI""","""728649""","""LTFM""","""ORBI""",2024-07-01 00:07:00,2024-07-01 02:51:17


In [5]:
df.select(
    pl.len().alias("rows"),
    pl.col("icao").is_not_null().sum().alias("rows_with_icao"),
    pl.col("icao").is_null().sum().alias("rows_missing_icao"),
)

rows,rows_with_icao,rows_missing_icao
u32,u32,u32
935934,932865,3069


In [11]:
from datetime import date, datetime


df.filter(pl.col("ACTUAL OFF BLOCK TIME") > datetime(2024,7,1,0,0,0))

AC Registration,icao,ADEP,ADES,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME
str,str,str,str,datetime[μs],datetime[μs]
"""GEUXF""","""400afe""","""LTAI""","""EGKK""",2024-07-01 19:47:30,2024-07-02 00:29:18
"""N57111""","""a754b5""","""KEWR""","""ESSA""",2024-07-01 00:09:00,2024-07-01 07:32:40
"""N217UA""","""a1d546""","""KEWR""","""LIMC""",2024-07-01 00:20:00,2024-07-01 07:31:00
"""N41140""","""a4dad4""","""KEWR""","""LEMG""",2024-07-01 00:24:00,2024-07-01 07:40:43
"""N799UA""","""aadb37""","""KEWR""","""EHAM""",2024-07-01 00:17:00,2024-07-01 06:44:57
…,…,…,…,…,…
"""FHZEN""","""39e48d""","""GABS""","""LFPO""",2024-07-01 00:35:00,2024-07-01 06:16:50
"""PKLEW""","""8a09df""","""GMMN""","""OEJN""",2024-07-01 00:21:42,2024-07-01 06:05:38
"""N761AJ""","""aa4548""","""KDFW""","""EIDW""",2024-07-01 01:18:00,2024-07-01 09:04:15


# Investigate results

In [ ]:
from datetime import date

from flights.evaluation.evaluation import main
outputs = main(test_src="model", gold_src="eurocontrol", test_dates=date(2024, 6, 1))
df = outputs[0][0][0]

Stats for model:eurocontrol: {'true_positive': 6151, 'false_positive': 86, 'false_negative': 185, 'precision': 0.9862113195446529, 'recall': 0.9708017676767676, 'f1': 0.9784458760836713, 'takeoff_airport_ident_match_pct': 0.9587059014794342, 'landing_airport_ident_match_pct': 0.9479759388717282, 'airport_ident_match_pct': 0.9107462201268086}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-07-08_23-55_2c7cf4d_flights_evaluation.csv


Python(48198) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [3]:
COLUMNS = [
    "icao",
    "first_message_time",
    "first_lat",
    "first_lon",
    "first_baro_altitude_ft",
    "last_message_time",
    "last_lat",
    "last_lon",
    "last_baro_altitude_ft",
    "takeoff_time_df1",
    "landing_time_df1",
    "takeoff_airport_ident",
    "landing_airport_ident",
    "takeoff_airport_ident_df1",
    "landing_airport_ident_df1",
    "same_takeoff_airport_ident",
    "same_landing_airport_ident",
    "same_airport_ident",
    'match_status'
]
df = df.select(COLUMNS)

In [6]:
import polars as pl
df.filter(pl.col("match_status") != "both")

icao,first_message_time,first_lat,first_lon,first_baro_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,takeoff_time_df1,landing_time_df1,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status
str,datetime[ms],f64,f64,i64,datetime[ms],f64,f64,i64,datetime[ms],datetime[ms],str,str,str,str,bool,bool,bool,str
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 02:59:00,2024-06-01 05:22:28,null,null,"""LKTB""","""LGIR""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 06:17:00,2024-06-01 08:48:47,null,null,"""LGIR""","""LKTB""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 10:12:00,2024-06-01 14:16:43,null,null,"""LKTB""","""HEMA""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 15:14:17,2024-06-01 19:11:48,null,null,"""HEMA""","""LKTB""",false,false,false,"""df1_only"""
"""008feb""",2024-06-01 03:01:36.410,49.154137,16.683373,null,2024-06-01 08:48:24.710,49.153107,16.673161,975,null,null,"""LKTB""","""LKTB""",null,null,false,false,false,"""df0_only"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""e494f9""",2024-06-01 09:22:53.390,38.58985,-9.861938,10625,2024-06-01 09:33:44.880,38.760956,-9.146835,525,null,null,null,null,null,null,false,false,false,"""df0_only"""
"""e49539""",2024-06-01 06:38:19.300,38.909617,-10.256409,18800,2024-06-01 06:56:33.550,38.762283,-9.146178,525,null,null,"""LPMT""","""LPPT""",null,null,false,false,false,"""df0_only"""
"""e80445""",null,null,null,null,null,null,null,null,2024-06-01 04:06:00,2024-06-01 15:05:00,null,null,"""SPIM""","""LEMD""",false,false,false,"""df1_only"""


In [7]:
df.filter(pl.col("icao") == "008feb")

icao,first_message_time,first_lat,first_lon,first_baro_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,takeoff_time_df1,landing_time_df1,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_takeoff_airport_ident,same_landing_airport_ident,same_airport_ident,match_status
str,datetime[ms],f64,f64,i64,datetime[ms],f64,f64,i64,datetime[ms],datetime[ms],str,str,str,str,bool,bool,bool,str
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 02:59:00,2024-06-01 05:22:28,null,null,"""LKTB""","""LGIR""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 06:17:00,2024-06-01 08:48:47,null,null,"""LGIR""","""LKTB""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 10:12:00,2024-06-01 14:16:43,null,null,"""LKTB""","""HEMA""",false,false,false,"""df1_only"""
"""008feb""",null,null,null,null,null,null,null,null,2024-06-01 15:14:17,2024-06-01 19:11:48,null,null,"""HEMA""","""LKTB""",false,false,false,"""df1_only"""
"""008feb""",2024-06-01 03:01:36.410,49.154137,16.683373,null,2024-06-01 08:48:24.710,49.153107,16.673161,975,null,null,"""LKTB""","""LKTB""",null,null,false,false,false,"""df0_only"""
"""008feb""",2024-06-01 10:17:50.370,49.148875,16.724083,1575,2024-06-01 19:16:32.580,49.153507,16.690798,null,null,null,"""LKTB""","""LKTB""",null,null,false,false,false,"""df0_only"""
